# Первый день минутных свечей ADA из S3

Notebook находит самый ранний дневной parquet для `ADAUSDT/1m` и загружает его в `df`.

In [2]:
import io
import os

import boto3
import pandas as pd
from dotenv import load_dotenv

load_dotenv()

required_env = [
    "YC_ENDPOINT",
    "YC_REGION",
    "YC_ACCESS_KEY_ID",
    "YC_SECRET_ACCESS_KEY",
]
missing_env = [name for name in required_env if not os.getenv(name)]
if missing_env:
    raise RuntimeError(
        "Не настроен доступ к S3. Добавьте в .env: "
        + ", ".join(missing_env)
    )

BUCKET = os.getenv("YC_BUCKET", "binance-data-downloader")
PREFIX = "raw/klines/symbol=ADAUSDT/interval=1m/"

s3 = boto3.client(
    "s3",
    endpoint_url=os.getenv("YC_ENDPOINT"),
    region_name=os.getenv("YC_REGION"),
    aws_access_key_id=os.getenv("YC_ACCESS_KEY_ID"),
    aws_secret_access_key=os.getenv("YC_SECRET_ACCESS_KEY"),
)

In [3]:
response = s3.list_objects_v2(Bucket=BUCKET, Prefix=PREFIX, MaxKeys=1)
objects = response.get("Contents", [])
if not objects:
    raise FileNotFoundError(f"В s3://{BUCKET}/{PREFIX} не найдены свечи")

first_key = objects[0]["Key"]
first_key

'raw/klines/symbol=ADAUSDT/interval=1m/date=2020-02-01/data.parquet'

In [4]:
body = s3.get_object(Bucket=BUCKET, Key=first_key)["Body"].read()
df = pd.read_parquet(io.BytesIO(body))

print(f"Источник: s3://{BUCKET}/{first_key}")
print(f"Размер: {df.shape}")
df.head()

Источник: s3://binance-data-downloader/raw/klines/symbol=ADAUSDT/interval=1m/date=2020-02-01/data.parquet
Размер: (1440, 12)


,timestamp,open_time,close_time,open,high,low,close,volume,quote_volume,trades,taker_buy_base,taker_buy_quote
0,2020-02-01 00:00:00+00:00,1580515200000,1580515259999,0.05383,0.05397,0.05377,0.05382,303154.0,16327.074219,46,97519.0,5251.351562
1,2020-02-01 00:01:00+00:00,1580515260000,1580515319999,0.05382,0.05387,0.05381,0.05381,267470.0,14401.816406,25,67058.0,3609.846191
2,2020-02-01 00:02:00+00:00,1580515320000,1580515379999,0.05385,0.05386,0.05377,0.05377,85914.0,4625.166016,20,61800.0,3327.760010
3,2020-02-01 00:03:00+00:00,1580515380000,1580515439999,0.05377,0.05382,0.05368,0.05378,455970.0,24504.648438,49,232076.0,12473.500000
4,2020-02-01 00:04:00+00:00,1580515440000,1580515499999,0.05376,0.05382,0.05362,0.05368,90562.0,4861.035645,33,38391.0,2062.146484


In [5]:
assert len(df) == 1440, f"Ожидалось 1440 минут, получено {len(df)}"
assert df["timestamp"].dt.strftime("%Y-%m-%d").nunique() == 1

df.info()
df

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1440 entries, 0 to 1439
Data columns (total 12 columns):
 #   Column           Non-Null Count  Dtype              
---  ------           --------------  -----              
 0   timestamp        1440 non-null   datetime64[ns, UTC]
 1   open_time        1440 non-null   int64              
 2   close_time       1440 non-null   int64              
 3   open             1440 non-null   float32            
 4   high             1440 non-null   float32            
 5   low              1440 non-null   float32            
 6   close            1440 non-null   float32            
 7   volume           1440 non-null   float32            
 8   quote_volume     1440 non-null   float32            
 9   trades           1440 non-null   int64              
 10  taker_buy_base   1440 non-null   float32            
 11  taker_buy_quote  1440 non-null   float32            
dtypes: datetime64[ns, UTC](1), float32(8), int64(3)
memory usage: 90.1 KB


,timestamp,open_time,close_time,open,high,low,close,volume,quote_volume,trades,taker_buy_base,taker_buy_quote
0,2020-02-01 00:00:00+00:00,1580515200000,1580515259999,0.05383,0.05397,0.05377,0.05382,303154.0,16327.074219,46,97519.0,5251.351562
1,2020-02-01 00:01:00+00:00,1580515260000,1580515319999,0.05382,0.05387,0.05381,0.05381,267470.0,14401.816406,25,67058.0,3609.846191
2,2020-02-01 00:02:00+00:00,1580515320000,1580515379999,0.05385,0.05386,0.05377,0.05377,85914.0,4625.166016,20,61800.0,3327.760010
3,2020-02-01 00:03:00+00:00,1580515380000,1580515439999,0.05377,0.05382,0.05368,0.05378,455970.0,24504.648438,49,232076.0,12473.500000
4,2020-02-01 00:04:00+00:00,1580515440000,1580515499999,0.05376,0.05382,0.05362,0.05368,90562.0,4861.035645,33,38391.0,2062.146484
...,...,...,...,...,...,...,...,...,...,...,...,...
1435,2020-02-01 23:55:00+00:00,1580601300000,1580601359999,0.05622,0.05625,0.05610,0.05612,94003.0,5283.629883,34,45341.0,2549.468262
1436,2020-02-01 23:56:00+00:00,1580601360000,1580601419999,0.05613,0.05623,0.05607,0.05610,168804.0,9479.619141,54,62493.0,3511.204834
1437,2020-02-01 23:57:00+00:00,1580601420000,1580601479999,0.05612,0.05625,0.05604,0.05613,133693.0,7507.635742,46,83705.0,4701.923340
1438,2020-02-01 23:58:00+00:00,1580601480000,1580601539999,0.05618,0.05619,0.05607,0.05611,110783.0,6217.505371,45,28445.0,1597.860229
